<a href="https://colab.research.google.com/github/yogeshwardev/CSA6102-Digital_forensics-/blob/main/lab_from_29_to_38.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datetime import datetime, timedelta

def parse_time(t):
    return datetime.strptime(t, "%Y-%m-%d %H:%M:%S")

def detect_bruteforce(events, threshold=5, window_minutes=2):
    """Detect brute-force login attempts: >= threshold failed logons (4625)
    for the same account within window_minutes, and report whether a
    successful logon (4624) followed."""
    events = sorted(events, key=lambda e: parse_time(e["timestamp"]))
    by_account = {}
    for e in events:
        by_account.setdefault(e["account"], []).append(e)
    results = {}
    for account, acc_events in by_account.items():
        failures = [e for e in acc_events if e["event_id"] == 4625]
        successes = [e for e in acc_events if e["event_id"] == 4624]
        flagged = False
        for i in range(len(failures)):
            window_start = parse_time(failures[i]["timestamp"])
            window_end = window_start + timedelta(minutes=window_minutes)
            count = sum(
                1 for f in failures
                if window_start <= parse_time(f["timestamp"]) <= window_end
            )
            if count >= threshold:
                flagged = True
                break

        if flagged:
            followed_by_success = bool(successes) and any(
                parse_time(s["timestamp"]) > parse_time(failures[-1]["timestamp"])
                for s in successes
            )
            results[account] = {
                "failed_attempts": len(failures),
                "followed_by_success": followed_by_success,
                "source_ips": sorted({f["source_ip"] for f in failures}),
            }
    return results

def test_experiment29():
    events = []
    base = datetime(2026, 1, 15, 3, 40, 0)
    for i in range(6):
        events.append({
            "event_id": 4625, "account": "Administrator",
            "timestamp": (base + timedelta(seconds=15 * i)).strftime("%Y-%m-%d %H:%M:%S"),
            "source_ip": "203.0.113.7",
        })
    events.append({
        "event_id": 4624, "account": "Administrator",
        "timestamp": (base + timedelta(seconds=100)).strftime("%Y-%m-%d %H:%M:%S"),
        "source_ip": "203.0.113.7",
    })
    events.append({
        "event_id": 4624, "account": "jsmith",
        "timestamp": "2026-01-15 09:00:00", "source_ip": "10.0.0.5",
    })
    results = detect_bruteforce(events)
    assert "Administrator" in results
    assert results["Administrator"]["failed_attempts"] == 6
    assert results["Administrator"]["followed_by_success"] is True
    assert results["Administrator"]["source_ips"] == ["203.0.113.7"]
    assert "jsmith" not in results
    print("Experiment 29: All test cases passed.")

test_experiment29()


Experiment 29: All test cases passed.


In [2]:
from datetime import datetime

def parse_usbstor(registry_data):
    """registry_data: {device_id: {"serial", "friendly_name",
    "first_connected", "last_connected"}}. Returns list of device dicts
    sorted by last_connected, most recent first."""
    devices = []
    for device_id, info in registry_data.items():
        entry = {"device_id": device_id}
        entry.update(info)
        devices.append(entry)
    devices.sort(
        key=lambda d: datetime.strptime(d["last_connected"], "%Y-%m-%d %H:%M:%S"),
        reverse=True,
    )
    return devices

def find_device_near_time(devices, target_time_str, window_minutes=30):
    """Return devices whose last_connected time is within window_minutes
    of target_time_str."""
    target = datetime.strptime(target_time_str, "%Y-%m-%d %H:%M:%S")
    matches = []
    for d in devices:
        last = datetime.strptime(d["last_connected"], "%Y-%m-%d %H:%M:%S")
        delta_minutes = abs((target - last).total_seconds()) / 60
        if delta_minutes <= window_minutes:
            matches.append(d)
    return matches

def test_experiment30():
    registry_data = {
        "USB\\VID_0781&PID_5567\\4C531001234": {
            "serial": "4C531001234",
            "friendly_name": "SanDisk Cruzer Blade",
            "first_connected": "2025-11-01 09:00:00",
            "last_connected": "2026-02-10 17:42:00",
        },
        "USB\\VID_090C&PID_1000\\A1002233": {
            "serial": "A1002233",
            "friendly_name": "Kingston DataTraveler",
            "first_connected": "2024-06-01 10:00:00",
            "last_connected": "2024-06-01 10:15:00",
        },
    }
    devices = parse_usbstor(registry_data)
    assert devices[0]["friendly_name"] == "SanDisk Cruzer Blade"
    matches = find_device_near_time(devices, "2026-02-10 17:35:00", window_minutes=30)
    matched_names = [m["friendly_name"] for m in matches]
    assert "SanDisk Cruzer Blade" in matched_names
    assert "Kingston DataTraveler" not in matched_names
    print("Experiment 30: All test cases passed.")

test_experiment30()


Experiment 30: All test cases passed.


In [3]:
import re

AUTH_LINE_RE = re.compile(
    r"(?P<result>Accepted|Failed) password for (?P<user>\S+) from (?P<ip>[\d.]+) port (?P<port>\d+)"
)

def parse_auth_log(lines):
    """Parse raw auth.log lines into structured dicts."""
    entries = []
    for line in lines:
        match = AUTH_LINE_RE.search(line)
        if match:
            entries.append({
                "result": match.group("result"),
                "user": match.group("user"),
                "ip": match.group("ip"),
                "port": int(match.group("port")),
                "raw": line,
            })
    return entries

def flag_suspicious_logins(entries, trusted_ips):
    """Flag successful logins from untrusted IPs, especially for root."""
    flagged = []
    for e in entries:
        if e["result"] == "Accepted" and e["ip"] not in trusted_ips:
            severity = "HIGH" if e["user"] == "root" else "MEDIUM"
            flagged.append({**e, "severity": severity})
    return flagged

def test_experiment31():
    log_lines = [
        "Jan 15 08:00:01 server sshd[1001]: Accepted password for deploy from 10.0.0.5 port 51100 ssh2",
        "Jan 15 03:12:01 server sshd[1233]: Failed password for root from 198.51.100.23 port 51320 ssh2",
        "Jan 15 03:12:05 server sshd[1234]: Accepted password for root from 198.51.100.23 port 51322 ssh2",
    ]
    trusted_ips = {"10.0.0.5"}
    entries = parse_auth_log(log_lines)
    assert len(entries) == 3
    flagged = flag_suspicious_logins(entries, trusted_ips)
    assert len(flagged) == 1
    assert flagged[0]["user"] == "root"
    assert flagged[0]["ip"] == "198.51.100.23"
    assert flagged[0]["severity"] == "HIGH"
    print("Experiment 31: All test cases passed.")

test_experiment31()


Experiment 31: All test cases passed.


In [4]:
from datetime import datetime

TS_FMT = "%Y-%m-%d %H:%M:%S"

def detect_timestomping(file_meta, change_gap_minutes=60):
    """file_meta: {"modified", "accessed", "changed", "born"} as timestamp
    strings. Returns (is_suspicious: bool, reasons: list[str])."""
    m = datetime.strptime(file_meta["modified"], TS_FMT)
    a = datetime.strptime(file_meta["accessed"], TS_FMT)
    c = datetime.strptime(file_meta["changed"], TS_FMT)
    b = datetime.strptime(file_meta["born"], TS_FMT)
    reasons = []
    if m < b:
        reasons.append("Modified time is earlier than Born (creation) time")
    if a < b:
        reasons.append("Accessed time is earlier than Born (creation) time")
    gap_minutes = abs((c - m).total_seconds()) / 60
    if gap_minutes > change_gap_minutes and c > m:
        reasons.append(
            f"MFT Changed time is {gap_minutes:.0f} minutes after Modified time — "
            "metadata may have been altered after the fact"
        )
    return (len(reasons) > 0, reasons)

def test_experiment32():
    normal_file = {
        "born": "2026-01-10 09:00:00",
        "modified": "2026-01-10 09:05:00",
        "accessed": "2026-01-12 14:00:00",
        "changed": "2026-01-10 09:05:00",
    }
    tampered_file = {
        "born": "2026-02-01 12:00:00",
        "modified": "2020-01-01 00:00:00",
        "accessed": "2026-02-01 12:00:00",
        "changed": "2026-02-01 12:03:00",
    }
    is_susp1, reasons1 = detect_timestomping(normal_file)
    assert is_susp1 is False
    is_susp2, reasons2 = detect_timestomping(tampered_file)
    assert is_susp2 is True
    assert any("Modified time is earlier" in r for r in reasons2)
    print("Experiment 32: All test cases passed.")

test_experiment32()


Experiment 32: All test cases passed.


In [5]:
from datetime import datetime, timedelta

PKT_FMT = "%Y-%m-%d %H:%M:%S"

def detect_port_scan(packets, port_threshold=10, window_seconds=30):
    """Detect (src_ip -> dst_ip) pairs that contact >= port_threshold
    distinct destination ports within window_seconds."""
    packets = sorted(packets, key=lambda p: datetime.strptime(p["timestamp"], PKT_FMT))
    by_pair = {}
    for p in packets:
        key = (p["src_ip"], p["dst_ip"])
        by_pair.setdefault(key, []).append(p)
    results = {}
    for key, pkts in by_pair.items():
        for i in range(len(pkts)):
            start = datetime.strptime(pkts[i]["timestamp"], PKT_FMT)
            end = start + timedelta(seconds=window_seconds)
            ports_in_window = {
                p["dst_port"] for p in pkts
                if start <= datetime.strptime(p["timestamp"], PKT_FMT) <= end
            }
            if len(ports_in_window) >= port_threshold:
                results[key] = {
                    "distinct_ports": len(ports_in_window),
                    "ports": sorted(ports_in_window),
                }
                break
    return results

def test_experiment33():
    packets = []
    base = datetime(2026, 3, 1, 10, 0, 0)
    for i, port in enumerate(range(20, 32)):
        packets.append({
            "src_ip": "203.0.113.99", "dst_ip": "10.0.0.10",
            "dst_port": port,
            "timestamp": (base + timedelta(seconds=2 * i)).strftime(PKT_FMT),
        })
    for i in range(5):
        packets.append({
            "src_ip": "10.0.0.20", "dst_ip": "10.0.0.30",
            "dst_port": 443,
            "timestamp": (base + timedelta(seconds=5 * i)).strftime(PKT_FMT),
        })
    results = detect_port_scan(packets, port_threshold=10, window_seconds=30)
    assert ("203.0.113.99", "10.0.0.10") in results
    assert results[("203.0.113.99", "10.0.0.10")]["distinct_ports"] >= 10
    assert ("10.0.0.20", "10.0.0.30") not in results
    print("Experiment 33: All test cases passed.")

test_experiment33()


Experiment 33: All test cases passed.


In [6]:
import math
from collections import Counter

def shannon_entropy(s):
    if not s:
        return 0.0
    counts = Counter(s)
    length = len(s)
    return -sum((c / length) * math.log2(c / length) for c in counts.values())

def detect_dns_tunneling(queries, length_threshold=20, entropy_threshold=3.5):
    """queries: list of fully-qualified domain name strings.
    Flags queries whose leftmost label is long AND high-entropy."""
    flagged = []
    for q in queries:
        label = q.split(".")[0]
        entropy = shannon_entropy(label)
        if len(label) >= length_threshold and entropy >= entropy_threshold:
            flagged.append({"query": q, "label_length": len(label), "entropy": round(entropy, 2)})
    return flagged

def test_experiment34():
    queries = [
        "www.google.com",
        "mail.office365.com",
        "a8f3k2j9x1p7q4z6w0n5r2t8y3.exfil-domain.com",
        "vpn.corporate-network.com",
        "9c2e7b1a4f8d3c6e0a5b9d2f7c1e4a8b3d6f9c2e5a8b1d4f.tunnel.example.net",
    ]
    flagged = detect_dns_tunneling(queries)
    flagged_domains = [f["query"] for f in flagged]
    assert "www.google.com" not in flagged_domains
    assert "mail.office365.com" not in flagged_domains
    assert "vpn.corporate-network.com" not in flagged_domains
    assert "a8f3k2j9x1p7q4z6w0n5r2t8y3.exfil-domain.com" in flagged_domains
    assert "9c2e7b1a4f8d3c6e0a5b9d2f7c1e4a8b3d6f9c2e5a8b1d4f.tunnel.example.net" in flagged_domains
    assert len(flagged) == 2
    print("Experiment 34: All test cases passed.")

test_experiment34()


Experiment 34: All test cases passed.


In [7]:
SIGNATURES = [
    {"sid": 1000001, "name": "Possible SQL Injection", "pattern": "union select"},
    {"sid": 1000002, "name": "Directory Traversal Attempt", "pattern": "../../../etc/passwd"},
    {"sid": 1000003, "name": "Nmap Scripting Engine User-Agent", "pattern": "nmap scripting engine"},
]

def scan_payloads(packets, signatures=SIGNATURES):
    """packets: list of dicts with a 'payload' string field.
    Returns list of alerts: {packet_index, sid, name}."""
    alerts = []
    for idx, pkt in enumerate(packets):
        payload_lower = pkt["payload"].lower()
        for sig in signatures:
            if sig["pattern"] in payload_lower:
                alerts.append({
                    "packet_index": idx,
                    "sid": sig["sid"],
                    "name": sig["name"],
                })
    return alerts

def test_experiment35():
    packets = [
        {"payload": "GET /products?id=1 HTTP/1.1"},
        {"payload": "GET /login?user=admin' UNION SELECT username,password FROM users-- HTTP/1.1"},
        {"payload": "GET /download?file=../../../etc/passwd HTTP/1.1"},
    ]
    alerts = scan_payloads(packets)
    alert_indices = {a["packet_index"] for a in alerts}
    assert 0 not in alert_indices
    assert 1 in alert_indices
    assert 2 in alert_indices
    assert any(a["name"] == "Possible SQL Injection" for a in alerts)
    assert any(a["name"] == "Directory Traversal Attempt" for a in alerts)
    print("Experiment 35: All test cases passed.")

test_experiment35()


Experiment 35: All test cases passed.


In [8]:
def recover_deleted_messages(sms_table):
    """sms_table: list of dicts with is_deleted (bool) and overwritten (bool).
    Returns a list of rows that are deleted but still recoverable."""
    recoverable = [
        row for row in sms_table
        if row["is_deleted"] and not row["overwritten"]
    ]
    return recoverable

def summarize_table(sms_table):
    active = [r for r in sms_table if not r["is_deleted"]]
    recoverable = recover_deleted_messages(sms_table)
    lost = [r for r in sms_table if r["is_deleted"] and r["overwritten"]]
    return {"active": len(active), "recoverable": len(recoverable), "permanently_lost": len(lost)}

def test_experiment36():
    sms_table = [
        {"rowid": 1, "address": "+1-555-0101", "body": "See you at 6pm", "date": "2026-01-01", "is_deleted": False, "overwritten": False},
        {"rowid": 2, "address": "+1-555-0199", "body": "Transfer the funds now, delete after reading", "date": "2026-01-02", "is_deleted": True, "overwritten": False},
        {"rowid": 3, "address": "+1-555-0150", "body": "Old spam message", "date": "2025-06-01", "is_deleted": True, "overwritten": True},
    ]
    recoverable = recover_deleted_messages(sms_table)
    assert len(recoverable) == 1
    assert recoverable[0]["rowid"] == 2
    assert "Transfer the funds" in recoverable[0]["body"]
    summary = summarize_table(sms_table)
    assert summary == {"active": 1, "recoverable": 1, "permanently_lost": 1}
    print("Experiment 36: All test cases passed.")

test_experiment36()


Experiment 36: All test cases passed.


In [9]:
from datetime import datetime
from collections import Counter

LOG_FMT = "%Y-%m-%d %H:%M:%S"

def build_baseline_ips(logs):
    """Return {user: most_common_ip} based on all log activity."""
    by_user = {}
    for entry in logs:
        by_user.setdefault(entry["user"], []).append(entry["ip"])
    return {user: Counter(ips).most_common(1)[0][0] for user, ips in by_user.items()}

def flag_anomalous_downloads(logs, business_start=8, business_end=20):
    baseline = build_baseline_ips(logs)
    flagged = []
    for entry in logs:
        if entry["action"] != "download":
            continue
        ts = datetime.strptime(entry["timestamp"], LOG_FMT)
        reasons = []
        if entry["ip"] != baseline.get(entry["user"]):
            reasons.append("IP differs from user baseline")
        if not (business_start <= ts.hour < business_end):
            reasons.append("Outside business hours")
        if reasons:
            flagged.append({**entry, "reasons": reasons})
    return flagged

def test_experiment37():
    logs = [
        {"user": "alice", "action": "view", "file": "roadmap.docx", "timestamp": "2026-03-01 10:00:00", "ip": "10.0.0.5"},
        {"user": "alice", "action": "view", "file": "budget.xlsx", "timestamp": "2026-03-02 11:00:00", "ip": "10.0.0.5"},
        {"user": "alice", "action": "download", "file": "report.pdf", "timestamp": "2026-03-03 14:00:00", "ip": "10.0.0.5"},
        {"user": "alice", "action": "download", "file": "customer_database.csv", "timestamp": "2026-03-05 02:15:00", "ip": "185.220.101.7"},
    ]
    flagged = flag_anomalous_downloads(logs)
    assert len(flagged) == 1
    assert flagged[0]["file"] == "customer_database.csv"
    assert "IP differs from user baseline" in flagged[0]["reasons"]
    assert "Outside business hours" in flagged[0]["reasons"]
    print("Experiment 37: All test cases passed.")

test_experiment37()


Experiment 37: All test cases passed.


In [10]:
import subprocess
import sys
import os

scripts = [
    "Exp-29-Windows-Event-Log-Brute-Force.py",
    "Exp-30-Windows-Registry-USB-History.py",
    "Exp-31-Linux-SSH-Auth-Log.py",
    "Exp-32-File-Timestamp-Timestomping.py",
    "Exp-33-Port-Scan-Detector.py",
    "Exp-34-DNS-Tunneling-Detector.py",
    "Exp-35-Signature-Based-IDS.py",
    "Exp-36-Mobile-SMS-Recovery.py",
    "Exp-37-Cloud-Audit-Log-Anomaly.py",
]

def run_all():
    print("Running all experiments 29 through 37...\n")
    all_passed = True
    for script in scripts:
        print(f"=== Running {script} ===")
        if not os.path.exists(script):
            print(f"Error: {script} not found in the current directory.")
            all_passed = False
            continue

        result = subprocess.run([sys.executable, script], capture_output=True, text=True)
        if result.returncode != 0:
            print(f"Error executing {script}:\n{result.stderr}")
            all_passed = False
        else:
            print(result.stdout)

    if all_passed:
        print("\nAll 9 experiments passed their test cases successfully.")
    else:
        print("\nSome experiments failed.")

if __name__ == "__main__":
    run_all()


Running all experiments 29 through 37...

=== Running Exp-29-Windows-Event-Log-Brute-Force.py ===
Error: Exp-29-Windows-Event-Log-Brute-Force.py not found in the current directory.
=== Running Exp-30-Windows-Registry-USB-History.py ===
Error: Exp-30-Windows-Registry-USB-History.py not found in the current directory.
=== Running Exp-31-Linux-SSH-Auth-Log.py ===
Error: Exp-31-Linux-SSH-Auth-Log.py not found in the current directory.
=== Running Exp-32-File-Timestamp-Timestomping.py ===
Error: Exp-32-File-Timestamp-Timestomping.py not found in the current directory.
=== Running Exp-33-Port-Scan-Detector.py ===
Error: Exp-33-Port-Scan-Detector.py not found in the current directory.
=== Running Exp-34-DNS-Tunneling-Detector.py ===
Error: Exp-34-DNS-Tunneling-Detector.py not found in the current directory.
=== Running Exp-35-Signature-Based-IDS.py ===
Error: Exp-35-Signature-Based-IDS.py not found in the current directory.
=== Running Exp-36-Mobile-SMS-Recovery.py ===
Error: Exp-36-Mobile-SM